# ⚙️ Feature Engineering

## Objective

The objective of this notebook is to create meaningful business features from the cleaned dataset.

These engineered features will improve machine learning model performance and provide deeper business insights.

### Features to Create

- Customer Lifetime
- Purchase Frequency
- Total Spending
- Average Basket Size
- Average Delivery Time
- Review Score
- Order Value
- Customer Recency
- Customer Tenure

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("cleaned_master_dataset.csv")

In [3]:
date_columns = [

    "order_purchase_timestamp",

    "order_delivered_customer_date"

]

for col in date_columns:
    df[col] = pd.to_datetime(df[col])

In [4]:
df["delivery_days"] = (

    df["order_delivered_customer_date"]

    -

    df["order_purchase_timestamp"]

).dt.days

In [5]:
df["delivery_days"].describe()

count    115722.000000
mean         12.022589
std           9.454922
min           0.000000
25%           6.000000
50%          10.000000
75%          15.000000
max         209.000000
Name: delivery_days, dtype: float64

In [6]:
customer_spending = (

    df.groupby("customer_unique_id")["price"]

    .sum()

    .reset_index()

)

customer_spending.columns = [

    "customer_unique_id",

    "total_spending"

]

In [7]:
df = df.merge(

    customer_spending,

    on="customer_unique_id",

    how="left"

)

In [8]:
purchase_frequency = (

    df.groupby("customer_unique_id")["order_id"]

    .nunique()

    .reset_index()

)

purchase_frequency.columns = [

    "customer_unique_id",

    "purchase_frequency"

]

In [9]:
df = df.merge(

    purchase_frequency,

    on="customer_unique_id"

)

In [10]:
average_order = (

    df.groupby("customer_unique_id")["price"]

    .mean()

    .reset_index()

)

average_order.columns = [

    "customer_unique_id",

    "average_order_value"

]

In [11]:
df = df.merge(

    average_order,

    on="customer_unique_id"

)

In [12]:
review = (

    df.groupby("customer_unique_id")["review_score"]

    .mean()

    .reset_index()

)

review.columns = [

    "customer_unique_id",

    "average_review_score"

]

In [13]:
df = df.merge(

    review,

    on="customer_unique_id"

)

In [14]:
customer_lifetime = (

    df.groupby("customer_unique_id")

    ["order_purchase_timestamp"]

    .agg(["min","max"])

)

In [15]:
customer_lifetime["customer_lifetime_days"]=(

customer_lifetime["max"]

-

customer_lifetime["min"]

).dt.days

In [16]:
customer_lifetime = customer_lifetime.reset_index()

In [17]:
customer_lifetime = customer_lifetime[

    [

        "customer_unique_id",

        "customer_lifetime_days"

    ]

]

In [18]:
df = df.merge(

    customer_lifetime,

    on="customer_unique_id"

)

In [19]:
reference_date = df["order_purchase_timestamp"].max()

In [20]:
last_purchase = (

    df.groupby("customer_unique_id")

    ["order_purchase_timestamp"]

    .max()

    .reset_index()

)

In [21]:
last_purchase["recency_days"]=(

reference_date

-

last_purchase["order_purchase_timestamp"]

).dt.days

In [22]:
df=df.merge(

last_purchase[

["customer_unique_id","recency_days"]

],

on="customer_unique_id"

)

In [23]:
df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,...,seller_city,seller_state,product_category_name_english,delivery_days,total_spending,purchase_frequency,average_order_value,average_review_score,customer_lifetime_days,recency_days
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,...,itaquaquecetuba,SP,office_furniture,8.0,124.99,1,124.99,4.0,0,474
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,...,itajai,SC,housewares,16.0,289.00,1,289.00,5.0,0,233
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,...,itaquaquecetuba,SP,office_furniture,26.0,139.94,1,139.94,5.0,0,106
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,...,itaquaquecetuba,SP,office_furniture,14.0,149.94,1,149.94,5.0,0,173
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,...,ibitinga,SP,home_confort,11.0,230.00,1,230.00,5.0,0,35


In [24]:
df.to_csv(

"feature_dataset.csv",

index=False

)